# exp-back — worked example 2: exp_back Preserves Shape Across Different Tensor Ranks

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `exp-back`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Backward functions must return a gradient with the same shape as their corresponding input — the chain rule is elementwise for elementwise ops. For `exp(x)`, regardless of whether `x` is a scalar, a vector, or a 2-D matrix, the backward result has the same shape as `x`. The operation `grad_out * out` naturally satisfies this because it is a pointwise product.

## Worked solution

We apply `exp_back` to inputs of three different ranks and verify shape preservation.

**Scalar (shape `()`):** `x = tensor(0.5)`. `out = exp(0.5) ≈ 1.649`. `exp_back(tensor(1.), out, x)` returns a scalar gradient with shape `()`.

**Matrix (shape `(2, 3)`):** Each element is exponentiated independently. The backward result is a `(2, 3)` matrix where each element is `grad_out[i,j] * out[i,j]`.

**Why this matters:** If your backward function accidentally introduced a reduction (e.g. summed the gradient), the shape would change and the accumulated gradients for the leaf parameters would be wrong.

In [ ]:
import torch as t
from torch import Tensor

t.manual_seed(14)

def exp_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    return grad_out * out

# Scalar case
x_scalar = t.tensor(0.5)
out_scalar = t.exp(x_scalar)
g_scalar = exp_back(t.ones_like(out_scalar), out_scalar, x_scalar)
print(f"Scalar: x shape {x_scalar.shape}, grad shape {g_scalar.shape}")
assert g_scalar.shape == x_scalar.shape == t.Size([])

# 1-D case
x_1d = t.randn(5)
out_1d = t.exp(x_1d)
g_1d = exp_back(t.ones_like(out_1d), out_1d, x_1d)
print(f"1-D:    x shape {x_1d.shape}, grad shape {g_1d.shape}")
assert g_1d.shape == x_1d.shape

# 2-D case
x_2d = t.randn(2, 3)
out_2d = t.exp(x_2d)
g_2d = exp_back(t.ones_like(out_2d), out_2d, x_2d)
print(f"2-D:    x shape {x_2d.shape}, grad shape {g_2d.shape}")
assert g_2d.shape == x_2d.shape

# Verify 2-D values match autograd
x_2d_ag = x_2d.clone().requires_grad_(True)
t.exp(x_2d_ag).sum().backward()
assert t.allclose(g_2d, x_2d_ag.grad)
print("Shape preservation and correctness confirmed for all ranks.")